# Applying the Neighbor-Joining Algorithm

The neighbor-joining algorithm is one of the most popular methods for evolutionary tree reconstruction. We will first use the neighbor-joining algorithm to construct a small tree by hand.

Below is a distance matrix D containing distances between four different organisms labeled $W, X, Y$, and $Z$.

**Input Distance Matrix $D$**:

$$
\begin{array}{c|cccc}
  & W & X & Y & Z \\
\hline
W & 0 & 11 & 2 & 16 \\
X & 11 & 0 & 13 & 15 \\
Y & 2 & 13 & 0 & 9 \\
Z & 16 & 15 & 9 & 0
\end{array}
$$

In [35]:
from IPython.display import display, Markdown

def get_total_distances(D):
    """Calculates the TotalDistance for each row in D."""
    return [sum(row) for row in D]

def calculate_neighbor_joining_matrix(D):
    """
    Constructs the neighbor-joining matrix D* from distance matrix D.
    D*[i][j] = (n - 2) * D[i][j] - TotalDistance(i) - TotalDistance(j)
    """
    n = len(D)
    total_dist = get_total_distances(D)
    D_star = [[0] * n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            if i != j:
                d_ij = D[i][j]
                d_star_val = (n - 2) * d_ij - total_dist[i] - total_dist[j]
                D_star[i][j] = d_star_val
            else:
                D_star[i][j] = 0
                
    return D_star

def find_closest_pair(D_star):
    """Finds the indices (i, j) with the minimum value in D*."""
    n = len(D_star)
    min_val = float('inf')
    best_pair = (-1, -1)
    
    for i in range(n):
        for j in range(i + 1, n):
            if D_star[i][j] < min_val:
                min_val = D_star[i][j]
                best_pair = (i, j)
    return best_pair[0], best_pair[1], min_val

def update_distance_matrix(D, labels, i, j, new_label):
    """
    Updates the distance matrix by joining nodes i and j into a new node.
    Returns the new distance matrix, new labels list, and limb lengths for i and j.
    """
    n = len(D)
    total_dist = get_total_distances(D)
    
    # Calculate Limb Lengths
    d_ij = D[i][j]
    delta = (total_dist[i] - total_dist[j]) / (n - 2)
    limb_i = (d_ij + delta) / 2
    limb_j = (d_ij - delta) / 2
    
    # Prepare new matrix
    new_size = n - 1
    new_D = [[0] * new_size for _ in range(new_size)]
    
    # Determine new labels and mapping
    # New node will be at index 0
    active_indices = [k for k in range(n) if k != i and k != j]
    new_labels = [new_label] + [labels[k] for k in active_indices]
    
    # Fill new matrix
    # Row 0 (New Node) distances to other nodes k
    for idx, k in enumerate(active_indices):
        # Distance from new node to k
        # D(new, k) = (D(i, k) + D(j, k) - D(i, j)) / 2
        d_ik = D[i][k]
        d_jk = D[j][k]
        d_new_k = (d_ik + d_jk - d_ij) / 2
        
        # New Matrix indices: 0 is new node, idx+1 is k
        new_D[0][idx+1] = d_new_k
        new_D[idx+1][0] = d_new_k
        
    # Rest of the matrix (distances between old active nodes)
    for idx1, k1 in enumerate(active_indices):
        for idx2, k2 in enumerate(active_indices):
            new_D[idx1+1][idx2+1] = D[k1][k2]
            
    return new_D, new_labels, limb_i, limb_j

def print_matrix(matrix, labels, title):
    # Construct LaTeX string
    latex_str = f"**{title}**\n\n"
    latex_str += "$$\n\\begin{array}{c|" + "c" * len(labels) + "}\n"
    
    # Header
    latex_str += "  & " + " & ".join(labels) + " \\\\\n"
    latex_str += "\\hline\n"
    
    # Rows
    for i, row in enumerate(matrix):
        formatted_row = []
        for x in row:
            if abs(x - round(x)) < 1e-9:
                formatted_row.append(str(int(round(x))))
            else:
                formatted_row.append(f"{x:.2f}")
        latex_str += f"{labels[i]} & " + " & ".join(formatted_row) + " \\\\\n"
        
    latex_str += "\\end{array}\n$$"
    display(Markdown(latex_str))
    return latex_str
    
D = [
    [0, 11, 2, 16],
    [11, 0, 13, 15],
    [2, 13, 0, 9],
    [16, 15, 9, 0]
]
labels = ['W', 'X', 'Y', 'Z']


### Step 1: Calculate $D^*$

The first step is to construct the neighbor-joining matrix $D^*$. The element $D^*_{i,j}$ is calculated as:

$$D^*_{i,j} = (n - 2) \cdot D_{i,j} - TotalDistance(i) - TotalDistance(j)$$

where $TotalDistance(i)$ is the sum of distances from node $i$ to all other nodes. We will implement helper functions to calculate this matrix.

In [36]:
# Calculate and Print D*
D_star = calculate_neighbor_joining_matrix(D)
latex = print_matrix(D_star, labels, "Neighbor-Joining Matrix D*:")
print(latex)
# Find closest pair
i, j, min_val = find_closest_pair(D_star)
print(f"\nMinimum value in D*: {min_val:.2f}")
print(f"Closest neighbors: {labels[i]} (index {i}) and {labels[j]} (index {j})")

**Neighbor-Joining Matrix D*:**

$$
\begin{array}{c|cccc}
  & W & X & Y & Z \\
\hline
W & 0 & -46 & -49 & -37 \\
X & -46 & 0 & -37 & -49 \\
Y & -49 & -37 & 0 & -46 \\
Z & -37 & -49 & -46 & 0 \\
\end{array}
$$

**Neighbor-Joining Matrix D*:**

$$
\begin{array}{c|cccc}
  & W & X & Y & Z \\
\hline
W & 0 & -46 & -49 & -37 \\
X & -46 & 0 & -37 & -49 \\
Y & -49 & -37 & 0 & -46 \\
Z & -37 & -49 & -46 & 0 \\
\end{array}
$$

Minimum value in D*: -49.00
Closest neighbors: W (index 0) and Y (index 2)


### Step 2: Calculate $D_2$

The minimum element in $D^*$ corresponds to the indices for $W$ and $Y$ (indices 0 and 2). This implies that $W$ and $Y$ are neighbors. We will join these nodes into a new internal node, which we will label $A$.

To construct the new distance matrix $D_2$, we first calculate the limb lengths from the new node $A$ to $W$ and $Y$ using the formula:

$$LimbLength(W) = \frac{D_{W,Y} + \Delta}{2}, \quad LimbLength(Y) = \frac{D_{W,Y} - \Delta}{2}$$

where $\Delta = \frac{TotalDistance(W) - TotalDistance(Y)}{n - 2}$.

Then, we compute the distances from the new node $A$ to the remaining nodes ($X$ and $Z$) to form the reduced matrix $D_2$. The distance from the new node $A$ to any other node $k$ is:

$$D_{A,k} = \frac{D_{W,k} + D_{Y,k} - D_{W,Y}}{2}$$

In [37]:
# Join nearest neighbors found in previous step (W and Y) -> Create Node A
# Note: i and j were determined in the previous cell to be 0 (W) and 2 (Y)
new_label = "A"
D2, labels2, limb_i, limb_j = update_distance_matrix(D, labels, i, j, new_label)

print(f"Limb Length {labels[i]}: {limb_i:.2f}")
print(f"Limb Length {labels[j]}: {limb_j:.2f}")

print("\n")
latex = print_matrix(D2, labels2, "Distance Matrix D2:")
print(latex)

Limb Length W: 2.25
Limb Length Y: -0.25




**Distance Matrix D2:**

$$
\begin{array}{c|ccc}
  & A & X & Z \\
\hline
A & 0 & 11 & 11.50 \\
X & 11 & 0 & 15 \\
Z & 11.50 & 15 & 0 \\
\end{array}
$$

**Distance Matrix D2:**

$$
\begin{array}{c|ccc}
  & A & X & Z \\
\hline
A & 0 & 11 & 11.50 \\
X & 11 & 0 & 15 \\
Z & 11.50 & 15 & 0 \\
\end{array}
$$


### Step 3: Calculate $D_2^*$

We now have a reduced $3 \times 3$ matrix $D_2$ with nodes $A, X, Z$. To proceed, we treat this as a new problem instance.

We calculate the new neighbor-joining matrix $D_2^*$ for these 3 nodes using the same formula as before. We then look for the minimum entry in $D_2^*$ to identify the next pair of neighbors to join.

In [38]:
# Calculate D2*
D2_star = calculate_neighbor_joining_matrix(D2)
latex = print_matrix(D2_star, labels2, "Neighbor-Joining Matrix D2*:")
print(latex)
# Find closest pair in D2*
i2, j2, min_val2 = find_closest_pair(D2_star)
print(f"\nMinimum value in D2*: {min_val2:.2f}")
print(f"Closest neighbors: {labels2[i2]} and {labels2[j2]}")

**Neighbor-Joining Matrix D2*:**

$$
\begin{array}{c|ccc}
  & A & X & Z \\
\hline
A & 0 & -37.50 & -37.50 \\
X & -37.50 & 0 & -37.50 \\
Z & -37.50 & -37.50 & 0 \\
\end{array}
$$

**Neighbor-Joining Matrix D2*:**

$$
\begin{array}{c|ccc}
  & A & X & Z \\
\hline
A & 0 & -37.50 & -37.50 \\
X & -37.50 & 0 & -37.50 \\
Z & -37.50 & -37.50 & 0 \\
\end{array}
$$

Minimum value in D2*: -37.50
Closest neighbors: A and X


### Step 4: Calculate $D_3$

The minimum entry in $D_2^*$ identifies $A$ and $X$ as the nearest neighbors. We join them into a new internal node $B$.

We calculate the limb lengths for $A$ and $X$, and then compute the distances for the new matrix $D_3$.

This results in a $2 \times 2$ matrix $D_3$ containing only the new node $B$ and the remaining leaf node $Z$. At this point, the tree topology is fully determined.

In [39]:
# Join A and X -> Create Node B
new_label_2 = "B"
D3, labels3, limb_i2, limb_j2 = update_distance_matrix(D2, labels2, i2, j2, new_label_2)

print(f"Limb Length {labels2[i2]}: {limb_i2:.2f}")
print(f"Limb Length {labels2[j2]}: {limb_j2:.2f}")

print("\n")
latex = print_matrix(D3, labels3, "Distance Matrix D3:")
print(latex)

Limb Length A: 3.75
Limb Length X: 7.25




**Distance Matrix D3:**

$$
\begin{array}{c|cc}
  & B & Z \\
\hline
B & 0 & 7.75 \\
Z & 7.75 & 0 \\
\end{array}
$$

**Distance Matrix D3:**

$$
\begin{array}{c|cc}
  & B & Z \\
\hline
B & 0 & 7.75 \\
Z & 7.75 & 0 \\
\end{array}
$$


### Step 5: Construct Phylogeny Tree

We have reduced the distance matrix to $2 \times 2$, containing the internal node $B$ and leaf node $Z$. The neighbor-joining process is complete, and the remaining edge connects $B$ and $Z$ with the distance given in $D_3$.

To finalize the tree, we combine all the edges determined in the previous steps:
1.  From Step 2: Edges $(W, A)$ and $(Y, A)$ with their respective limb lengths.
2.  From Step 4: Edges $(A, B)$ and $(X, B)$ with their respective limb lengths.
3.  From Step 5: The final edge $(B, Z)$ with distance $D_{3_{0,1}}$.

We output the final adjacency list representing the constructed phylogeny tree.

In [41]:
# Initialize Adjacency List
adj = {}

def add_edge(u, v, w):
    if u not in adj: adj[u] = []
    if v not in adj: adj[v] = []
    adj[u].append((v, w))
    adj[v].append((u, w))

# Edges from Step 2 (W, Y joined to A)
# limb_i corresponds to W (labels[i]), limb_j corresponds to Y (labels[j])
add_edge(labels[i], 'A', limb_i)
add_edge(labels[j], 'A', limb_j)

# Edges from Step 4 (A, X joined to B)
# limb_i2 corresponds to A (labels2[i2]), limb_j2 corresponds to X (labels2[j2])
add_edge(labels2[i2], 'B', limb_i2)
add_edge(labels2[j2], 'B', limb_j2)

# Final Edge (B joined to Z)
# The remaining nodes in D3/labels3 are B and Z. Distance is in D3.
dist_B_Z = D3[0][1]
add_edge(labels3[0], labels3[1], dist_B_Z)

# Print Adjacency List
# Sorting keys for consistent output
print("Final Adjacency List:")
for u in adj.keys():
    neighbors = adj[u]
    for v, w in neighbors:
        print(f"{u}->{v}:{w:.2f}")

Final Adjacency List:
W->A:2.25
A->W:2.25
A->Y:-0.25
A->B:3.75
Y->A:-0.25
B->A:3.75
B->X:7.25
B->Z:7.75
X->B:7.25
Z->B:7.75


In [42]:
def get_path_distance(adj, start, end):
    """Finds the distance between two nodes in the tree using BFS."""
    queue = [(start, 0.0)]
    visited = set()
    while queue:
        curr, dist = queue.pop(0)
        if curr == end: return dist
        visited.add(curr)
        if curr in adj:
            for neighbor, weight in adj[curr]:
                if neighbor not in visited:
                    queue.append((neighbor, dist + weight))
    return float('inf')

print("\nValidation: Comparison of Original vs. Tree Distances")
print("-" * 50)
print(f"{'Pair':<10} {'Original':<10} {'Tree (Calc)':<15} {'Difference':<10}")
print("-" * 50)

total_error = 0
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        u, v = labels[i], labels[j]
        orig_dist = D[i][j]
        tree_dist = get_path_distance(adj, u, v)
        diff = tree_dist - orig_dist
        total_error += abs(diff)
        print(f"{u}-{v:<8} {orig_dist:<10} {tree_dist:<15.2f} {diff:<10.2f}")

print("-" * 50)
print(f"Total Absolute Error (Residuals): {total_error:.2f}")


Validation: Comparison of Original vs. Tree Distances
--------------------------------------------------
Pair       Original   Tree (Calc)     Difference
--------------------------------------------------
W-X        11         13.25           2.25      
W-Y        2          2.00            0.00      
W-Z        16         13.75           -2.25     
X-Y        13         10.75           -2.25     
X-Z        15         15.00           0.00      
Y-Z        9          11.25           2.25      
--------------------------------------------------
Total Absolute Error (Residuals): 9.00
